In [1]:
#这个版本成功的使用了presistent但是没有搞好ms缓存，现在我要处理多场景问题了，先保存一下

In [ ]:
# ===================== ALL-IN-ONE: 3D simplex (tetrahedra) for PID-SP with gurobi_persistent =====================
# Requirements: pyomo, gurobi, numpy, scipy, plotly, tqdm, csv file "data.csv"
# -----------------------------------------------------------------------------------------

import numpy as np
import itertools as it
import csv
from tqdm import tqdm
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
from pyomo.solvers.plugins.solvers.gurobi_persistent import GurobiPersistent
from scipy.spatial import Delaunay
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# ===== debug bucket =====
LAST_DEBUG = None   # 如果 next_node == 顶点 或 撞车，会把当时的上下文塞进来

# ------------------------- Config knobs -------------------------
MIN_DIST   = 1e-8     # 去重阈值
ACTIVE_TOL = 1e-8     # active 判定容差
MS_AGG     = "sum"    # 单形 ms 聚合：'sum' 或 'mean'

# ------------------------- PID scenario model -------------------------
def build_pid_model(T=10, h=0.2, scen=None, weights=(1.0, 0.01),
                    bounds=None, use_cvar=False, alpha=0.95):
    assert scen is not None, "请提供一个场景字典"
    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d) == T+1 and len(sp) == T+1

    if bounds is None:
        bounds = {}
    bx = bounds.get("x",  (-20, 20))
    bu = bounds.get("u",  (None, None))
    bKp= bounds.get("Kp", (0, 10))
    bKi= bounds.get("Ki", (0, 10))
    bKd= bounds.get("Kd", (0, 10))
    be = bounds.get("e",  (-100, 100))
    bI = bounds.get("I",  (-200, 200))

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    m.Kp = pyo.Var(bounds=bKp)
    m.Ki = pyo.Var(bounds=bKi)
    m.Kd = pyo.Var(bounds=bKd)

    m.x = pyo.Var(m.T, bounds=bx)
    m.u = pyo.Var(m.T, bounds=bu)
    m.e = pyo.Var(m.T, bounds=be)
    m.I = pyo.Var(m.T, bounds=bI)

    # error
    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    # integral
    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    # plant
    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    # pid
    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))
    m.obj_expr = pyo.Expression(expr=m.cost)  # 只保留表达式，目标在持久化阶段统一创建
    return m, [m.Kp, m.Ki, m.Kd]

def load_scenarios_from_csv(csv_path: str, T: int | None = None,
                            sp0: float = 0.0, sp1: float = 0.5,
                            ku_col: str = "tau_us", tau_col: str = "tau_xs",
                            disturb_prefix: str = "disturbance_",
                            setpoint_change_col: str = "setpoint_change"):
    scens = []
    # 推断 T
    if T is None:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fields  = reader.fieldnames or []
            max_idx = -1
            for name in fields:
                if name.startswith(disturb_prefix):
                    try:
                        k = int(name[len(disturb_prefix):])
                        max_idx = max(max_idx, k)
                    except:
                        pass
            if max_idx < 0:
                raise ValueError(f"未找到扰动列前缀 {disturb_prefix}k")
            T = max_idx

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            Ku  = float(row[ku_col])
            tau = float(row[tau_col])

            d = []
            for t in range(T+1):
                col = f"{disturb_prefix}{t}"
                d.append(float(row[col]))

            sp = [sp1]*(T+1)
            if setpoint_change_col in row and row[setpoint_change_col] != "":
                try:
                    t_star = int(float(row[setpoint_change_col]))
                    for t in range(T+1):
                        sp[t] = sp0 if t < t_star else sp1
                except:
                    pass

            scens.append({"Ku": Ku, "tau": tau, "d": d, "sp": sp})
    return scens, T

def build_models_from_csv(csv_path: str, h: float = 0.2,
                          weights=(1.0, 0.01), bounds=None,
                          sp0: float = 0.0, sp1: float = 0.5,
                          ku_col: str = "tau_us", tau_col: str = "tau_xs",
                          disturb_prefix: str = "disturbance_",
                          setpoint_change_col: str = "setpoint_change",
                          max_scenarios=None, skip=0):
    scens, T = load_scenarios_from_csv(
        csv_path=csv_path, T=None, sp0=sp0, sp1=sp1,
        ku_col=ku_col, tau_col=tau_col,
        disturb_prefix=disturb_prefix,           # ✅ FIX 这里：用 disturb_prefix（参数），不是 disturbance_
        setpoint_change_col=setpoint_change_col,
    )
    if skip or max_scenarios:
        scens = scens[skip: (skip + max_scenarios) if max_scenarios else None]

    model_list, first_stg_vars_list = [], []
    for scen in scens:
        m, yvars = build_pid_model(T=T, h=h, scen=scen, weights=weights, bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(yvars)

    m_tmpl_list = [model_list[0], first_stg_vars_list[0]]
    return model_list, first_stg_vars_list, m_tmpl_list, T


# ------------------------- Persistent wrappers -------------------------
class BaseBundle:
    """每个场景的基础模型（计算真实Q）+ 持久化求解器"""
    def __init__(self, model: pyo.ConcreteModel, options: dict | None = None):
        self.model = model
        self.gp = GurobiPersistent()
        self.gp.set_instance(model)
        if hasattr(model, 'obj'):
            model.del_component('obj')
        model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)
        self.gp.set_objective(model.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

    def eval_at(self, first_vars, first_vals):
        for v, val in zip(first_vars, first_vals):
            v.fix(float(val))
            self.gp.update_var(v)
        self.gp.solve(load_solutions=True)
        val = float(pyo.value(self.model.obj_expr))
        for v in first_vars:
            v.unfix()
            self.gp.update_var(v)
        return val
class MSBundle:
    """ms 模板（持久化）：每次更新四面体时 remove+add 约束，兼容没有
    set_linear_coefficient / set_constraint_expr 的 Pyomo 版本。
    """
    def __init__(self, model_base: pyo.ConcreteModel, first_vars, options: dict | None = None):
        # 克隆该场景模型
        m = model_base.clone()

        # 重心变量与和=1
        m.lam_index = pyo.RangeSet(0, 3)
        m.lam = pyo.Var(m.lam_index, domain=pyo.NonNegativeReals)
        m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in m.lam_index) == 1.0)

        # 一阶段变量引用
        self.Kp = m.find_component(first_vars[0].name)
        self.Ki = m.find_component(first_vars[1].name)
        self.Kd = m.find_component(first_vars[2].name)
        if any(v is None for v in (self.Kp, self.Ki, self.Kd)):
            raise RuntimeError("克隆模型中找不到 Kp/Ki/Kd")

        # 先放占位约束（系数0），真正的表达式在 update_tetra 里重建
        m.link_kp = pyo.Constraint(expr=self.Kp == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_ki = pyo.Constraint(expr=self.Ki == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_kd = pyo.Constraint(expr=self.Kd == sum(0.0 * m.lam[j] for j in m.lam_index))

        m.As = pyo.Var()
        m.As_def = pyo.Constraint(expr=m.As == sum(0.0 * m.lam[j] for j in m.lam_index))

        # 目标
        if hasattr(m, 'obj'):
            m.del_component('obj')
        m.obj = pyo.Objective(expr=m.obj_expr - m.As, sense=pyo.minimize)

        # 持久化求解器
        self.model = m
        self.gp = GurobiPersistent()
        self.gp.set_instance(m)
        self.gp.set_objective(m.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

        # 缓存/别名
        self.lam = m.lam
        self.link_kp = m.link_kp
        self.link_ki = m.link_ki
        self.link_kd = m.link_kd
        self.As     = m.As
        self.As_def = m.As_def
        self._V_cached = None  # [(x,y,z)]*4

    # ---- 工具：构建表达式 & 热替换约束（先从求解器移除，再从模型删掉，重建并 add 回去） ----
    def _expr_link(self, lhs_var, coeffs):
        return lhs_var == sum(float(coeffs[j]) * self.lam[j] for j in range(4))

    def _replace_constraint(self, attr_name: str, expr):
        # 1) 从求解器里移除旧约束（若已存在）
        old_con = getattr(self.model, attr_name)
        try:
            self.gp.remove_constraint(old_con)
        except Exception:
            pass  # 第一次可能还没被单独 add 过

        # 2) 从模型里删除旧组件
        self.model.del_component(old_con)

        # 3) 以相同名字重建新约束并加入模型
        new_con = pyo.Constraint(expr=expr)
        self.model.add_component(attr_name, new_con)

        # 4) 通知持久化求解器新增该约束
        self.gp.add_constraint(getattr(self.model, attr_name))

    # ---- 外部接口：写入四面体数据并求解 ----
    def update_tetra(self, tet_vertices, fverts_scene):
        # 稳定排序（与外层一致）
        pairs = sorted(
            [(tuple(map(float, tet_vertices[j])), float(fverts_scene[j])) for j in range(4)],
            key=lambda kv: (kv[0][0], kv[0][1], kv[0][2])
        )
        V = [kv[0] for kv in pairs]  # 顶点坐标
        F = [kv[1] for kv in pairs]  # 顶点值
        self._V_cached = V

        vx = [V[j][0] for j in range(4)]
        vy = [V[j][1] for j in range(4)]
        vz = [V[j][2] for j in range(4)]

        # 用 remove+add 的方式重建四条等式
        self._replace_constraint('link_kp', self._expr_link(self.Kp, vx))
        self._replace_constraint('link_ki', self._expr_link(self.Ki, vy))
        self._replace_constraint('link_kd', self._expr_link(self.Kd, vz))
        self._replace_constraint('As_def',  self._expr_link(self.As, F))

    def solve(self):
        res = self.gp.solve(load_solutions=True)
        ok = (res.solver.status == SolverStatus.ok) and \
             (res.solver.termination_condition in {
                 TerminationCondition.optimal,
                 TerminationCondition.locallyOptimal
             })
        return ok

    def get_ms_and_point(self):
        ms_val = float(pyo.value(self.model.obj))
        lam_star = np.array([pyo.value(self.lam[j]) for j in range(4)], dtype=float)
        V = np.array(self._V_cached, dtype=float)
        new_pt = lam_star @ V
        return ms_val, lam_star, tuple(map(float, new_pt))


def build_persistent_bundles(model_list, first_stg_vars_list, gurobi_options: dict):
    base_bundles = []
    ms_bundles   = []
    for m, yvars in zip(model_list, first_stg_vars_list):
        base_bundles.append(BaseBundle(m, gurobi_options))
        ms_bundles.append(MSBundle(m, yvars, gurobi_options))
    return base_bundles, ms_bundles

# ------------------------- Basic utils -------------------------
def corners_from_var_bounds(vars_3):
    bnds = []
    for v in vars_3:
        lb, ub = v.lb, v.ub
        if lb is None or ub is None:
            raise ValueError(f"{v.name} 缺少上下界")
        bnds.append((float(lb), float(ub)))
    return [tuple(p) for p in it.product(*[(lo, hi) for (lo,hi) in bnds])]

def too_close(p, nodes, tol=MIN_DIST):
    return any(np.linalg.norm(np.asarray(p)-np.asarray(q)) < tol for q in nodes)

def evaluate_Q_at(base_bundle: BaseBundle, first_stg_vars, first_stg_vals):
    return base_bundle.eval_at(first_stg_vars, first_stg_vals)

def tet_volume(verts):
    V = np.array(verts, float)
    v0, v1, v2, v3 = V
    return float(abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0)

def tet_quality(verts):
    V = np.array(verts, float)
    edges = [np.linalg.norm(V[i] - V[j]) for (i, j) in it.combinations(range(4), 2)]
    denom = float(np.sum(np.power(edges, 3))) + 1e-16
    vol = tet_volume(verts)
    return float(6.0 * vol / denom)

# ------------------------- Single tetra & scene: ms solve (persistent) -------------------------
def ms_on_tetra_for_scene(ms_bundle: MSBundle, tet_vertices, fverts_scene):
    ms_bundle.update_tetra(tet_vertices, fverts_scene)
    ok = ms_bundle.solve()
    if not ok:
        return float('inf'), None, None
    ms_val, lam_star, new_pt = ms_bundle.get_ms_and_point()
    return ms_val, lam_star, new_pt

# ------------------------- Evaluate all tetrahedra -------------------------
def evaluate_all_tetra(nodes, scen_values, ms_bundles, first_vars_list):
    pts = np.asarray(nodes, dtype=float)
    if len(pts) < 4:
        return None, []
    tri = Delaunay(pts)
    S = len(ms_bundles)

    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    diam = float(np.linalg.norm(maxs - mins))
    vol_tol = 1e-12 * max(diam**3, 1.0)

    per_tet = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        verts = [tuple(pts[i]) for i in idxs]

        v0, v1, v2, v3 = np.array(verts)
        vol = abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0
        if vol < vol_tol:
            continue

        fverts_per_scene = [[scen_values[ω][i] for i in idxs] for ω in range(S)]
        fverts_sum = [sum(fverts_per_scene[ω][j] for ω in range(S)) for j in range(4)]

        ms_scene = []
        xms_scene = []
        for ω in range(S):
            ms_val, lam_star, new_pt = ms_on_tetra_for_scene(
                ms_bundles[ω], verts, fverts_per_scene[ω]
            )
            ms_scene.append(ms_val)
            xms_scene.append(new_pt)

        if MS_AGG == "sum":
            ms_total = float(np.sum(ms_scene))
        elif MS_AGG == "mean":
            ms_total = float(np.mean(ms_scene))
        else:
            raise ValueError("MS_AGG must be 'sum' or 'mean'")

        LB = float(np.min(fverts_sum) + ms_total)
        UB = float(np.max(fverts_sum) + ms_total)

        best_scene = int(np.argmin(ms_scene))
        x_ms_best = xms_scene[best_scene]

        per_tet.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "fverts_sum": fverts_sum,
            "ms_per_scene": ms_scene,
            "ms": ms_total,
            "LB": LB,
            "UB": UB,
            "x_ms_best_scene": x_ms_best,
            "best_scene": best_scene,
            "volume": vol,
        })

    return tri, per_tet

# ------------------------- Pretty print -------------------------
def print_tetra_table(per_tet, active_mask, purple_set=None, prec=6):
    purple_set = set() if purple_set is None else set(purple_set)
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    tet_ids = [r["simplex_index"] for r in per_tet]
    active_set = {tid for tid in tet_ids if active_mask.get(tid, False)}

    def _mark(tid):
        s = f"T{tid}"
        flags = []
        if tid in active_set:  flags.append("*")
        if tid in purple_set:  flags.append("^")
        return s + ("".join(flags) if flags else "")

    header = ["row\\simp"] + [_mark(tid) for tid in tet_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.{prec}f}" for r in per_tet],
        ["LB"] + [f"{r['LB']:.{prec}f}" for r in per_tet],
        ["ms"] + [f"{r['ms']:.3e}"       for r in per_tet],
    ]
    table = [header] + rows
    colw = [max(len(str(row[c])) for row in table) + 2 for c in range(len(header))]

    RED, PURPLE, RESET = "\033[31m", "\033[35m", "\033[0m"
    def colorize(col_idx, s):
        if col_idx == 0:
            return s
        tid = tet_ids[col_idx-1]
        if tid in purple_set:
            return f"{PURPLE}{s}{RESET}"
        elif tid in active_set:
            return f"{RED}{s}{RESET}"
        return s

    print("\n== Per-tetra summary ==")
    print("".join(colorize(c, str(header[c]).ljust(colw[c])) for c in range(len(header))))
    print("-"*sum(colw))
    for r in rows:
        line = []
        for c in range(len(header)):
            cell = str(r[c])
            pad  = cell.ljust(colw[c]) if c==0 else cell.rjust(colw[c])
            line.append(colorize(c, pad))
        print("".join(line))
    print("(红色列=active；紫色列=包含当前最小节点的单形；第1行=UB，第2行=LB，第3行=ms)\n")

def min_dist_to_nodes(pt, nodes):
    P = np.asarray(pt, float)
    X = np.asarray(nodes, float)
    return float(np.min(np.linalg.norm(X - P, axis=1)))

def print_per_scenario_ms(per_tet, max_scenarios_to_print=10, prec=3):
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    if not per_tet or "ms_per_scene" not in per_tet[0]:
        return
    S = len(per_tet[0]["ms_per_scene"])
    show = min(S, max_scenarios_to_print)
    head = "simp | " + " ".join([f"s{j}".rjust(10) for j in range(show)])
    print("== Per-tetra per-scenario ms (showing first", show, "of", S, "scenes) ==")
    print(head); print("-"*len(head))
    for r in per_tet:
        arr = r["ms_per_scene"][:show]
        sline = " ".join([f"{v:.{prec}e}".rjust(10) for v in arr])
        print(f"{r['simplex_index']:>4d} | {sline}")
    if show < S:
        print(f"... ({S-show} scenes omitted)")
    print()

# ------------------------- Plotly visualization -------------------------
def plot_iteration_plotly(iter_id, nodes, tri, active_mask, ub_node, next_node, per_tet,
                          highlight_simplices=None):
    import numpy as np
    import plotly.graph_objects as go

    if highlight_simplices is None:
        highlight_simplices = set()
    else:
        highlight_simplices = set(highlight_simplices)

    fig = go.Figure()
    nodes = np.asarray(nodes, float)

    if len(nodes) > 0:
        fig.add_trace(go.Scatter3d(
            x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2],
            mode='markers',
            marker=dict(size=4, color="black"),
            name='nodes'
        ))

    if ub_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[ub_node[0]], y=[ub_node[1]], z=[ub_node[2]],
            mode='markers',
            marker=dict(size=7, symbol="circle", color="green"),
            name='current min node'
        ))

    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode='markers',
            marker=dict(size=8, symbol="diamond", color="#1976d2"),
            name='next node'
        ))

    def _is_same_point(a, b, atol=1e-6):
        if a is None or b is None:
            return False
        return np.linalg.norm(np.asarray(a, float) - np.asarray(b, float)) <= float(atol)

    if tri is not None:
        legend_mesh_added = False
        legend_edge_added = False

        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue

            verts = np.array(r["verts"], dtype=float)
            highlight_by_next = _is_same_point(next_node, r.get("x_ms_best_scene", None), atol=1e-6)
            highlight = highlight_by_next or (sid in highlight_simplices)

            mesh_color = "#ff5722" if highlight else "#ffb74d"
            edge_color = "darkorange"
            edge_width = 4 if highlight else 3
            mesh_opacity = 0.45 if highlight else 0.35

            I = [0, 0, 0, 1]
            J = [1, 1, 2, 2]
            K = [2, 3, 3, 3]

            fig.add_trace(go.Mesh3d(
                x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                i=I, j=J, k=K,
                color=mesh_color,
                opacity=mesh_opacity,
                showscale=False,
                name="active simplex",
                showlegend=(not legend_mesh_added)
            ))
            legend_mesh_added = True

            edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
            for (a, b) in edges:
                pa, pb = verts[a], verts[b]
                fig.add_trace(go.Scatter3d(
                    x=[pa[0], pb[0]],
                    y=[pa[1], pb[1]],
                    z=[pa[2], pb[2]],
                    mode='lines',
                    line=dict(width=edge_width, color=edge_color),
                    name='active edge',
                    showlegend=(not legend_edge_added)
                ))
            legend_edge_added = True

            cx, cy, cz = np.mean(verts, axis=0)
            qtxt = ""
            if "quality" in r and r["quality"] is not None:
                try:
                    qtxt = f"<br>q={float(r['quality']):.3e}"
                except Exception:
                    qtxt = ""
            txt = (f"simp={sid}"
                   f"<br>LB={float(r['LB']):.6f}"
                   f"<br>UB={float(r['UB']):.6f}"
                   f"<br>ms={float(r['ms']):.3e}"
                   f"<br>vol={float(r['volume']):.3e}"
                   f"{qtxt}")

            fig.add_trace(go.Scatter3d(
                x=[cx], y=[cy], z=[cz],
                mode='markers',
                marker=dict(size=1, opacity=0.0),
                text=[txt], hoverinfo="text",
                name="tetra info",
                showlegend=False
            ))

    fig.update_layout(
        title=f"Iteration {iter_id}",
        scene=dict(
            xaxis_title="Kp",
            yaxis_title="Ki",
            zaxis_title="Kd",
            aspectmode="cube",
            zaxis=dict(tickformat=".2f"),
        ),
        width=980,
        height=720,
        legend=dict(itemsizing="constant")
    )
    fig.update_traces(
        hovertemplate="x: %{x:.6f}<br>y: %{y:.6f}<br>z: %{z:.6f}",
        selector=dict(type='scatter3d')
    )
    fig.show()

# ------------------------- MAIN LOOP -------------------------
def run_pid_simplex_3d(base_bundles, ms_bundles, model_list, first_vars_list,
                       target_nodes=30, min_dist=MIN_DIST, active_tol=ACTIVE_TOL, verbose=True):
    global LAST_DEBUG
    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []
    UB_node_hist, add_node_hist = [], []
    ms_a_hist, ms_b_hist = [], []
    active_ratio_hist = []

    S = len(model_list)
    nodes = corners_from_var_bounds(first_vars_list[0])

    bounds_arr = np.array([[float(v.lb), float(v.ub)] for v in first_vars_list[0]], float)
    diam = float(np.linalg.norm(bounds_arr[:,1] - bounds_arr[:,0]))
    min_dist = float(min_dist)

    # 缓存 f_ω(node_i)
    scen_values = [[None]*len(nodes) for _ in range(S)]
    for i, node in enumerate(nodes):
        for ω in range(S):
            scen_values[ω][i] = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], node)

    it = 0
    stop_due_to_collision = False
    while len(nodes) < target_nodes:
        # 1) 全局 UB
        f_sum_per_node = [
            sum(scen_values[ω][i] for ω in range(S))
            for i in range(len(nodes))
        ]
        ub_idx = int(np.argmin(f_sum_per_node))
        UB_global = float(f_sum_per_node[ub_idx])
        UB_node = tuple(nodes[ub_idx])

        # 2) 评估所有四面体（使用持久化 ms）
        tri, per_tet = evaluate_all_tetra(nodes, scen_values, ms_bundles, first_vars_list)
        if tri is None or not per_tet:
            if verbose:
                print("Not enough nodes to make tetrahedra; stop.")
            break

        # 3) active mask（按 UB 过滤 + 形状质量）
        active_mask = {
            r["simplex_index"]: (r["LB"] <= UB_global + active_tol)
            for r in per_tet
        }
        q_cut = 1e-3
        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue
            q = tet_quality(r["verts"])
            if q < q_cut:
                active_mask[sid] = False

        # 4) active ratio
        total_vol = sum(r["volume"] for r in per_tet)
        active_vol = sum(r["volume"] for r in per_tet if active_mask[r["simplex_index"]])
        active_ratio = active_vol / total_vol if total_vol > 0 else 0.0

        # 5) LB_global & ms_b
        ub_active = [r for r in per_tet
                     if (ub_idx in r["vert_idx"]) and active_mask.get(r["simplex_index"], False)]
        if ub_active:
            ms_b_rec   = min(ub_active, key=lambda r: r["ms"])
            ms_b       = float(ms_b_rec["ms"])
            ms_b_simp  = int(ms_b_rec["simplex_index"])
            LB_global  = UB_global + ms_b
        else:
            ms_b       = float('nan')
            ms_b_simp  = None
            active_LBs = [r["LB"] for r in per_tet if active_mask.get(r["simplex_index"], False)]
            LB_global  = float(min(active_LBs)) if active_LBs else float(min(r["LB"] for r in per_tet))

        # 6) ms_a（active 内最小 ms）
        if any(active_mask.values()):
            ms_a = float(min(r["ms"] for r in per_tet if active_mask[r["simplex_index"]]))
        else:
            ms_a = float(min(r["ms"] for r in per_tet))
        ms_iter = ms_a

        # 7) 记录
        LB_hist.append(LB_global)
        UB_hist.append(UB_global)
        ms_hist.append(ms_iter)
        node_count.append(len(nodes))
        UB_node_hist.append(UB_node)
        ms_a_hist.append(ms_a)
        ms_b_hist.append(ms_b)
        active_ratio_hist.append(active_ratio)

        # 8) 打印
        simp_with_min = [r["simplex_index"] for r in per_tet if ub_idx in r["vert_idx"]]
        if verbose:
            print(f"[Iter {it}] Active simplex ratio = {active_ratio:.6f}")
            print(f"[Iter {it}] UB node {UB_node} is in simplices {sorted(simp_with_min)}")
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")

        # 9) 候选排行
        active = [r for r in per_tet if active_mask[r["simplex_index"]]]
        ub_active = [r for r in active if ub_idx in r["vert_idx"]]
        candidates_pool = ub_active if len(ub_active) > 0 else active

        def score(cand):
            ms = cand["ms"]
            pt = cand.get("x_ms_best_scene", None)
            d  = (float('inf') if pt is None else min_dist_to_nodes(pt, nodes))
            return (ms, -d)

        candidates_sorted = sorted(candidates_pool, key=score)
        if verbose:
            top_msg = "N/A"
            if len(candidates_sorted) > 0:
                top0 = candidates_sorted[0]
                top_msg = f"T{int(top0['simplex_index'])}, ms={float(top0['ms']):.3e}"
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")
            print(f"[Iter {it}] candidate rank #1: {top_msg}")

            topN = candidates_sorted[:10]
            print("== ms candidates (sorted by (ms, -dist)) ==")
            print(f"{'rank':>4} {'simp':>6} {'ms':>12} {'mind(all)':>12} {'pt':>30}")
            print("-" * 90)
            for rnk, cand in enumerate(topN, start=1):
                pt = cand.get("x_ms_best_scene", None)
                d  = (float('nan') if pt is None else min_dist_to_nodes(pt, nodes))
                pt_str = "None" if pt is None else f"({pt[0]:.4f}, {pt[1]:.4f}, {pt[2]:.4f})"
                print(f"{rnk:>4} T{cand['simplex_index']:<4} {cand['ms']:>12.4e} {d:>12.2e} {pt_str:>30}")
            print()

        # 10) 选新点 + 强校验/撞车处理
        new_node = None
        chosen_ms = None
        chosen_cand = None
        stop_due_to_collision = False

        def handle_collision(cand_pt, cand, stage_note="active"):
            nonlocal stop_due_to_collision
            X = np.asarray(nodes, float)
            P = np.asarray(cand_pt, float)
            dists = np.linalg.norm(X - P, axis=1)
            j_star = int(np.argmin(dists))
            d_star = float(dists[j_star])
            orange_ids = [r["simplex_index"] for r in per_tet if j_star in r["vert_idx"]]
            debug_pack = {
                "reason": "candidate_too_close",
                "iter": it,
                "stage": stage_note,
                "min_dist": float(min_dist),
                "closest_node_index": j_star,
                "closest_node_point": tuple(map(float, nodes[j_star])),
                "closest_distance": d_star,
                "cand_simplex": int(cand["simplex_index"]),
                "cand_point": tuple(map(float, cand_pt)),
                "cand_ms": float(cand["ms"]),
                "UB_global": float(UB_global),
                "LB_global": float(LB_global),
                "active_ratio": float(active_ratio),
                "UB_node": tuple(map(float, UB_node)),
                "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                "per_tet_snapshot": [
                    {
                        "simplex_index": int(r["simplex_index"]),
                        "vert_idx": list(map(int, r["vert_idx"])),
                        "verts": [tuple(map(float, x)) for x in r['verts']],
                        "ms": float(r["ms"]),
                        "LB": float(r["LB"]),
                        "UB": float(r["UB"]),
                        "best_scene": int(r["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])),
                        "volume": float(r["volume"]),
                    } for r in per_tet
                ],
                "highlight_simplices": list(map(int, orange_ids)),
            }
            global LAST_DEBUG
            LAST_DEBUG = debug_pack
            plot_iteration_plotly(
                it, nodes, tri, active_mask, UB_node, cand_pt, per_tet,
                highlight_simplices=orange_ids
            )
            if verbose:
                print(
                    f"[STOP] Candidate {tuple(map(float, cand_pt))} "
                    f"is too close to existing node #{j_star} at distance {d_star:.3e} "
                    f"(< {min_dist:g}). Highlighted simplices: {sorted(orange_ids)}"
                )
            stop_due_to_collision = True

        for rank, cand in enumerate(candidates_sorted, start=1):
            cand_pt = cand.get("x_ms_best_scene", None)
            if cand_pt is None:
                continue
            if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                new_node   = cand_pt
                chosen_ms  = cand["ms"]
                chosen_cand= cand
                if verbose:
                    print(
                        f"Chosen node {tuple(map(float, cand_pt))} "
                        f"with ms={chosen_ms:.3e} "
                        f"(simp T{cand['simplex_index']}, rank #{rank})"
                    )
                    print(f"[Iter {it}] next node comes from simplex T{int(cand['simplex_index'])}")
                break
            else:
                if verbose:
                    print(
                        f"Skip candidate {tuple(map(float, cand_pt))} "
                        f"(simp T{cand['simplex_index']}, rank #{rank}) "
                        f"because too close to existing nodes (< {min_dist:g})."
                    )
                handle_collision(cand_pt, cand, stage_note="active")
                break

        if (new_node is None) and (not stop_due_to_collision) and (len(active) > 0):
            if verbose:
                print("[fallback] All active candidates too close; try all simplices...")
            all_sorted = sorted(per_tet, key=score)
            for cand in all_sorted:
                cand_pt = cand.get("x_ms_best_scene", None)
                if cand_pt is None:
                    continue
                if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                    new_node   = cand_pt
                    chosen_ms  = cand["ms"]
                    chosen_cand= cand
                    if verbose:
                        print(
                            f"Chosen node {tuple(map(float, cand_pt))} "
                            f"with ms={chosen_ms:.3e} "
                            f"(simp T{cand['simplex_index']}) [fallback-all]"
                        )
                        print(f"[Iter {it}] next node comes from simplex T{int(cand['simplex_index'])} [fallback-all]")
                    break
                else:
                    if verbose:
                        print(
                            f"Skip (all) candidate {tuple(map(float, cand_pt))} "
                            f"(simp T{cand['simplex_index']}) "
                            f"because too close to existing nodes (< {min_dist:g})."
                        )
                    handle_collision(cand_pt, cand, stage_note="fallback-all")
                    break

        if stop_due_to_collision:
            if verbose:
                print(f"[Iter {it}] Stop due to collision.")
            break

        if new_node is None:
            if verbose:
                print("New node too close for all candidates (or infeasible ms); stop.")
            break

        # == 强校验 ==
        tol_same = 1e-10
        def _same(a, b, tol=tol_same):
            a = np.asarray(a, float); b = np.asarray(b, float)
            return np.linalg.norm(a - b) <= tol

        if chosen_cand is not None:
            offending_vert = None
            for v in chosen_cand["verts"]:
                if _same(new_node, v):
                    offending_vert = tuple(map(float, v))
                    break
            if offending_vert is not None:
                LAST_DEBUG = {
                    "reason": "next_node_equals_vertex",
                    "iter": it,
                    "new_node": tuple(map(float, new_node)),
                    "offending_vertex": offending_vert,
                    "tol_same": tol_same,
                    "candidate": {
                        "simplex_index": int(chosen_cand["simplex_index"]),
                        "vert_idx": list(map(int, chosen_cand["vert_idx"])),

                        "verts": [tuple(map(float, x)) for x in chosen_cand["verts"]],
                        "ms": float(chosen_cand["ms"]),
                        "ms_per_scene": [float(x) for x in chosen_cand["ms_per_scene"]],
                        "best_scene": int(chosen_cand["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, chosen_cand["x_ms_best_scene"])),
                        "LB": float(chosen_cand["LB"]),
                        "UB": float(chosen_cand["UB"]),
                        "volume": float(chosen_cand["volume"]),
                    },
                    "UB_global": float(UB_global),
                    "LB_global": float(LB_global),
                    "active_ratio": float(active_ratio),
                    "UB_node": tuple(map(float, UB_node)),
                    "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                    "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                    "per_tet_snapshot": [
                        {
                            "simplex_index": int(r["simplex_index"]),
                            "vert_idx": list(map(int, r["vert_idx"])),
                            "verts": [tuple(map(float, x)) for x in r["verts"]],
                            "ms": float(r["ms"]),
                            "LB": float(r["LB"]),
                            "UB": float(r["UB"]),
                            "best_scene": int(r["best_scene"]),
                            "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])),
                            "volume": float(r["volume"]),
                        } for r in per_tet
                    ],
                }
                vert_idx_list = []
                for j, nd in enumerate(nodes):
                    if _same(offending_vert, nd):
                        vert_idx_list.append(j)
                orange_ids = [r["simplex_index"] for r in per_tet if any(j in r["vert_idx"] for j in vert_idx_list)]
                plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                                      highlight_simplices=orange_ids)
                if verbose:
                    print("[STOP] new_node coincides with a simplex vertex. Highlighted simplices:",
                          sorted(orange_ids))
                break

        # 可视化（正常迭代）
        plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                              highlight_simplices=None)

        # 加点并评估（持久化 base）
        new_vals = []
        for ω in range(S):
            val = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], new_node)
            new_vals.append(val)

        nodes.append(tuple(map(float, new_node)))
        for ω in range(S):
            scen_values[ω].append(new_vals[ω])

        add_node_hist.append(new_node)
        it += 1

    return {
        "nodes": np.array(nodes, float),
        "LB_hist": LB_hist,
        "UB_hist": UB_hist,
        "ms_hist": ms_hist,
        "ms_a_hist": ms_a_hist,
        "ms_b_hist": ms_b_hist,
        "node_count": node_count,
        "UB_node_hist": UB_node_hist,
        "added_nodes": add_node_hist,
        "active_ratio_hist": active_ratio_hist,
    }

# ===================== MAIN =====================
RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 1
    target_nodes   = 30
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# build models
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# gurobi 参数
gurobi_options = {
    'MIPGap': 1e-1,
    'NumericFocus': 1,
    'Presolve': 2,
    'NonConvex': 2,   # 必须
    'TimeLimit': 10,  # 可按需打开/删除
}

# 持久化封装
base_bundles, ms_bundles = build_persistent_bundles(model_list, first_stg_vars_list, gurobi_options)

# run
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True
)

print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")
# ================================================================================================================


Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
[Iter 0] Active simplex ratio = 1.000000
[Iter 0] UB node (1.0, 1.0, 1.0) is in simplices [2, 3]
[Iter 0] LB = 0.162061 = UB(0.361626) + ms_b(-1.996e-01) from T2
[Iter 0] LB = 0.162061 = UB(0.361626) + ms_b(-1.996e-01) from T2
[Iter 0] candidate rank #1: T2, ms=-1.996e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T2     -1.9957e-01     5.23e-01       (1.0000, 0.3698, 0.3698)
   2 T3     -1.9957e-01     5.23e-01       (1.0000, 0.3698, 0.3698)

Chosen node (0.9999999820786589,

[Iter 1] Active simplex ratio = 1.000000
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 8, 9, 10, 11]
[Iter 1] LB = -0.558282 = UB(0.361626) + ms_b(-9.199e-01) from T2
[Iter 1] LB = -0.558282 = UB(0.361626) + ms_b(-9.199e-01) from T2
[Iter 1] candidate rank #1: T2, ms=-9.199e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T2     -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   2 T3     -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   3 T11    -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   4 T10    -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)

Chosen node (0.3116794718137422, 0.31167946840807353, 0.9999999984121736) with ms=-9.199e-01 (simp T2, rank #1)
[Iter 1] next node comes from simplex T2


[Iter 2] Active simplex ratio = 0.905630
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 10, 11, 16, 17]
[Iter 2] LB = 0.071659 = UB(0.361626) + ms_b(-2.900e-01) from T11
[Iter 2] LB = 0.071659 = UB(0.361626) + ms_b(-2.900e-01) from T11
[Iter 2] candidate rank #1: T11, ms=-2.900e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T11    -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   2 T10    -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   3 T5     -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)
   4 T4     -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)

Chosen node (0.3213541091906367, 0.999999993093919, 0.3213541134696063) with ms=-2.900e-01 (simp T11, rank #1)
[Iter 2] next node comes from simplex T11


[Iter 3] Active simplex ratio = 0.863803
[Iter 3] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 4, 5, 11, 12, 17, 18, 21, 22]
[Iter 3] LB = 0.222111 = UB(0.361626) + ms_b(-1.395e-01) from T4
[Iter 3] LB = 0.222111 = UB(0.361626) + ms_b(-1.395e-01) from T4
[Iter 3] candidate rank #1: T4, ms=-1.395e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T4     -1.3952e-01     3.72e-01       (0.4631, 0.6153, 0.8478)
   2 T5     -1.3952e-01     3.72e-01       (0.4631, 0.6153, 0.8478)
   3 T2     -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)
   4 T3     -5.7121e-02     4.20e-01       (1.0000, 0.8296, 0.3836)

Chosen node (0.4631080022651556, 0.615295912439594, 0.847812104697146) with ms=-1.395e-01 (simp T4, rank #1)
[Iter 3] next node comes from simplex T4


[Iter 4] Active simplex ratio = 0.826469
[Iter 4] UB node (1.0, 1.0, 1.0) is in simplices [0, 1, 2, 7, 9, 10, 21, 22, 25, 26, 27, 28]
[Iter 4] LB = 0.225722 = UB(0.361626) + ms_b(-1.359e-01) from T0
[Iter 4] LB = 0.225722 = UB(0.361626) + ms_b(-1.359e-01) from T0
[Iter 4] candidate rank #1: T0, ms=-1.359e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T0     -1.3590e-01     1.92e-01       (0.5709, 0.5709, 1.0000)
   2 T1     -1.3590e-01     1.92e-01       (0.5709, 0.5709, 1.0000)
   3 T7     -1.1924e-01     4.02e-01       (1.0000, 0.4023, 1.0000)
   4 T9     -5.7121e-02     4.20e-01       (1.0000, 0.8296, 0.3836)
   5 T10    -5.0312e-02     4.47e-01       (0.6991, 0.8481, 0.5473)
   6 T2     -4.8200e-02     3.85e-01       (0.5506, 0.9283, 0.6223)

Chosen node (0.5709144915305622, 0.570914376305411, 0.9999999952342482) with 

...

[Iter 9] Active simplex ratio = 0.690107
[Iter 9] UB node (1.0, 1.0, 1.0) is in simplices [0, 11, 12, 14, 15, 16, 17, 24, 28, 29, 34, 35, 37, 39, 45, 46, 49, 50, 51, 52]
[Iter 9] LB = 0.313426 = UB(0.361626) + ms_b(-4.820e-02) from T16
[Iter 9] LB = 0.313426 = UB(0.361626) + ms_b(-4.820e-02) from T16
[Iter 9] candidate rank #1: T16, ms=-4.820e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T16    -4.8200e-02     3.85e-01       (0.5506, 0.9283, 0.6223)
   2 T0     -4.8200e-02     3.85e-01       (0.5506, 0.9283, 0.6223)
   3 T29    -2.9708e-02     3.18e-01       (0.6983, 0.7989, 0.7376)
   4 T12    -2.9469e-02     3.02e-01       (0.5151, 0.8707, 1.0000)
   5 T24    -2.8305e-02     3.70e-01       (0.8526, 0.7054, 0.8019)
   6 T28    -2.8126e-02     3.30e-01       (0.7869, 0.7352, 0.8130)
   7 T35    -1.1172e-02     2.62e-01   

[Iter 10] Active simplex ratio = 0.660963
[Iter 10] UB node (1.0, 1.0, 1.0) is in simplices [10, 12, 13, 14, 21, 24, 25, 27, 29, 31, 39, 40, 42, 44, 50, 51, 54, 55, 56, 57]
[Iter 10] LB = 0.314856 = UB(0.361626) + ms_b(-4.677e-02) from T25
[Iter 10] LB = 0.314856 = UB(0.361626) + ms_b(-4.677e-02) from T25
[Iter 10] candidate rank #1: T25, ms=-4.677e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T25    -4.6770e-02     7.23e-02       (0.5546, 1.0000, 0.6141)
   2 T24    -4.6640e-02     8.82e-02       (0.5920, 1.0000, 0.5920)
   3 T29    -2.9469e-02     3.02e-01       (0.5151, 0.8707, 1.0000)
   4 T21    -2.8305e-02     3.70e-01       (0.8526, 0.7054, 0.8019)
   5 T31    -2.8126e-02     3.30e-01       (0.7869, 0.7352, 0.8130)
   6 T27    -1.1380e-02     2.81e-01       (0.9144, 0.9303, 0.6823)
   7 T40    -1.1172e-02     2.62e

[Iter 11] Active simplex ratio = 0.653642
[Iter 11] UB node (1.0, 1.0, 1.0) is in simplices [3, 5, 7, 18, 19, 26, 32, 33, 35, 37, 39, 40, 41, 42, 57, 58, 59, 60, 63, 64]
[Iter 11] LB = 0.332157 = UB(0.361626) + ms_b(-2.947e-02) from T5
[Iter 11] LB = 0.332157 = UB(0.361626) + ms_b(-2.947e-02) from T5
[Iter 11] candidate rank #1: T5, ms=-2.947e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T5     -2.9469e-02     3.02e-01       (0.5151, 0.8707, 1.0000)
   2 T19    -2.8305e-02     3.70e-01       (0.8526, 0.7054, 0.8019)
   3 T40    -2.8126e-02     3.30e-01       (0.7869, 0.7352, 0.8130)
   4 T39    -2.5864e-02     2.53e-01       (0.7029, 0.7769, 0.9359)
   5 T42    -1.1380e-02     2.81e-01       (0.9144, 0.9303, 0.6823)
   6 T33    -1.1172e-02     2.62e-01       (1.0000, 0.9215, 0.6511)
   7 T41    -1.1093e-02     3.05e-01   

[Iter 12] Active simplex ratio = 0.644479
[Iter 12] UB node (1.0, 1.0, 1.0) is in simplices [3, 4, 15, 16, 20, 21, 28, 34, 35, 37, 39, 41, 42, 43, 44, 45, 61, 62, 63, 64, 67, 68]
[Iter 12] LB = 0.333322 = UB(0.361626) + ms_b(-2.830e-02) from T21
[Iter 12] LB = 0.333322 = UB(0.361626) + ms_b(-2.830e-02) from T21
[Iter 12] candidate rank #1: T21, ms=-2.830e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T21    -2.8305e-02     3.70e-01       (0.8526, 0.7054, 0.8019)
   2 T41    -2.8126e-02     3.30e-01       (0.7869, 0.7352, 0.8130)
   3 T15    -2.6100e-02     1.55e-01       (0.4292, 1.0000, 1.0000)
   4 T43    -2.5304e-02     2.61e-01       (0.7552, 0.7552, 1.0000)
   5 T45    -1.1380e-02     2.81e-01       (0.9144, 0.9303, 0.6823)
   6 T35    -1.1172e-02     2.62e-01       (1.0000, 0.9215, 0.6511)
   7 T44    -1.1093e-02    

[Iter 13] Active simplex ratio = 0.622491
[Iter 13] UB node (1.0, 1.0, 1.0) is in simplices [3, 4, 15, 16, 20, 25, 31, 33, 35, 37, 38, 39, 40, 41, 43, 44, 67, 68, 69, 70, 73, 74]
[Iter 13] LB = 0.334036 = UB(0.361626) + ms_b(-2.759e-02) from T41
[Iter 13] LB = 0.334036 = UB(0.361626) + ms_b(-2.759e-02) from T41
[Iter 13] candidate rank #1: T41, ms=-2.759e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T41    -2.7590e-02     1.61e-01       (1.0000, 0.6515, 0.7664)
   2 T15    -2.6100e-02     1.55e-01       (0.4292, 1.0000, 1.0000)
   3 T40    -2.5329e-02     2.15e-01       (0.7776, 0.7448, 1.0000)
   4 T38    -2.5304e-02     2.26e-01       (0.7552, 0.7552, 1.0000)
   5 T44    -1.2209e-02     2.59e-01       (0.9507, 0.9015, 0.6636)
   6 T43    -1.2209e-02     2.59e-01       (0.9507, 0.9015, 0.6636)
   7 T39    -1.1858e-02    